DAY 2

In [ ]:
!pip install -q openai

In [ ]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Why did the scarecrow become a stand‑up comedian?

Because he was outstanding in his field… and finally got a *crow*d of laughs! 😄


In [ ]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We need at least 24 hours’ notice for any cake, so we can’t prepare an eggless cake for this evening. Let us know your preferred flavor and we’ll have it ready for you tomorrow!
CompletionUsage(completion_tokens=159, prompt_tokens=183, total_tokens=342, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=107, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.223039442, prompt_time=0.007011647, completion_time=0.336114, total_time=0.343125647)


In [ ]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}   # key → value

In [ ]:
MENU["donut"]                  # 500
MENU.get("donut", "donuts not available")   # 'not on menu'  (no crash)
#MENU["donut"]

'donuts not available'

In [ ]:
order = {"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": True}


1000

In [ ]:
order["qty"] * MENU[order["item"]] # 1000


1000

In [ ]:
import json
text = json.dumps(order)           # dict → JSON string, to send or save
back = json.loads(text)

In [ ]:
text

'{"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": true}'

In [ ]:
back

{'name': 'Ravi', 'item': 'plum cake', 'qty': 2, 'eggless': True}

In [ ]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [ ]:
def get_price(item, qty=1) :
    """Return the total price in rupees for qty of an item on the Sweet Crumbs menu."""
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [ ]:
a=90
print(a+b)
print('hi')

NameError: name 'b' is not defined

In [ ]:
try:
  a=10
  print(a+b)
  print('hi')
except:
  print('b is not defined')


b is not defined


In [ ]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "llama-3.3-70b-versatile"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: what are the languages your are aware of
Bot: We understand and can reply in many languages—English, Telugu, Hindi, Spanish, French, German, Mandarin, and more. Let us know your preferred language, and we’ll chat with you in it!
You: cupcakes unnaya
Bot: అవును, మాకు కప్‌కేక్స్ ఉన్నాయి—ప్రతి ఒక్కటి ₹80. అన్ని కేక్‌లలా ఎగ్‌లెస్ వేరియంట్ కూడా అందుబాటులో ఉంది. దయచేసి 24 గంటల ముందుగా ఆర్డర్ ఇవ్వండి. మీరు ఏ రుచి లేదా పరిమాణం కావాలో చెప్పండి!
You: quit
Bot: Thank you for visiting Sweet Crumbs!


In [ ]:
history

[{'role': 'system',
  'content': 'ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.\nFACTS:  Menu with prices in rupees: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n        Orders need 24 hours\' notice. All cakes have eggless versions.\nRULES:  Never promise a delivery time. If it isn\'t in FACTS, say "I\'ll check with the owner."\n        Remember the customer\'s name and order during the chat.\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'hi'},
 {'role': 'assistant',
  'content': 'Hi! 😊 Welcome to Sweet Crumbs Home Bakery. How can I help you today? May I have your name, please?'},
 {'role': 'user', 'content': 'plum cakes are availble'},
 {'role': 'assistant',
  'content': 'Yes, our plum cake is available at ₹500 each (eggless version also). 😊 Would you like to place an order? May I have your name, please?'},
 {'role': 'user', 'content': 'what are the items in your menu'},
 {'role': 'assis

## Prompt parts: remove the rules

Compare the answer with the RULES section removed and then restored.


In [ ]:
system_without_rules = SYSTEM.split("RULES:", 1)[0] + "FORMAT:" + SYSTEM.split("FORMAT:", 1)[1]
question = "Can you deliver in 1 hour?"

def ask_with_system(system_prompt, user_question):
    result = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_question}],
    )
    return result.choices[0].message.content

print("Without RULES:", ask_with_system(system_without_rules, question))
print("With RULES:", ask_with_system(SYSTEM, question))


## Few-shot business classifier

Use three labeled bakery examples, then classify five new customer messages.


In [ ]:
examples = [
    ("I want to order two plum cakes.", "ORDER"),
    ("How much is a cupcake?", "MENU_QUESTION"),
    ("Tell me a joke.", "OTHER"),
]
new_messages = [
    "Please reserve a chocolate cake for me.",
    "Are your cakes available eggless?",
    "How much do six cupcakes cost?",
    "Can you deliver in one hour?",
    "Do you sell donuts?",
]
classifier_messages = [
    {"role": "system", "content": "Classify bakery customer messages as ORDER, MENU_QUESTION, or OTHER. Reply with one label only."}
]
for example, label in examples:
    classifier_messages.extend([
        {"role": "user", "content": example},
        {"role": "assistant", "content": label},
    ])
classifier_messages.append({
    "role": "user",
    "content": "Classify each of these five messages in order, one label per line:\n" + "\n".join(new_messages),
})
classifier_response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=classifier_messages,
)
for message, label in zip(new_messages, classifier_response.choices[0].message.content.splitlines()):
    print(message, "->", label)


## JSON order extraction

Ask for a JSON order, parse it with `json.loads`, and handle invalid JSON explicitly.


In [ ]:
import json

order_request = "Ravi wants 2 eggless plum cakes for 2026-12-01."
json_response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": "Extract the order. Return only valid JSON with exactly these keys: name (string), item (string), qty (integer), date (YYYY-MM-DD string), eggless (boolean). Do not add markdown."},
        {"role": "user", "content": order_request},
    ],
)
json_text = json_response.choices[0].message.content
try:
    parsed_order = json.loads(json_text)
except json.JSONDecodeError as error:
    print("The model did not return valid JSON:", error)
    print("Raw response:", json_text)
else:
    print(parsed_order)


## Limited memory

Keep only the latest two exchanges. Give the bot your name, have three more turns, and then ask for your name. The first exchange should have fallen out of the model context.


In [ ]:
MAX_TURNS = 2
limited_history = [{"role": "system", "content": SYSTEM}]

def limited_memory_chat(user_text):
    limited_history.append({"role": "user", "content": user_text})
    recent_messages = [limited_history[0]] + limited_history[1:][-MAX_TURNS * 2:]
    result = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=recent_messages,
    )
    reply = result.choices[0].message.content
    limited_history.append({"role": "assistant", "content": reply})
    print("You:", user_text)
    print("Bot:", reply)
    return reply

limited_memory_chat("My name is Ravi.")
limited_memory_chat("Do you make eggless cakes?")
limited_memory_chat("How much is a chocolate cake?")
limited_memory_chat("How much notice do you need?")
limited_memory_chat("What is my name?")


With `MAX_TURNS = 2`, the request includes only the last two user/assistant exchanges plus the system prompt. After three intervening turns, the original name statement is no longer in the request, so the model may not know the name.


## Sweet Crumbs business setup

A complete business cell: menu dictionary, documented price function, and bakery-specific system prompt.


In [ ]:
import json

BUSINESS_MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

def business_get_price(item, qty=1):
    """Return the total price in rupees for a menu item and quantity."""
    return BUSINESS_MENU[item.lower().strip()] * qty

BUSINESS_SYSTEM = f"""ROLE: You are the assistant for Sweet Crumbs Home Bakery.
FACTS: Menu and prices in rupees: {json.dumps(BUSINESS_MENU)}. Orders need 24 hours' notice. All cakes have eggless versions.
RULES: Never promise a delivery time. If an answer is not in FACTS, say "I'll check with the owner." Keep replies friendly and under 50 words."""

print(business_get_price("cupcake", 6))
print(BUSINESS_SYSTEM)
